# Exercise 14-1 - Transfer Learning using the ImageNet Dataset

## Fine tune a pre-trained network

Here is another example for you to experiment with. The code below first reads in a set of jpeg files containing images. The code then loads a pre-trained network (Inception V3) and trains the final few fully-connected layers on the new dataset after freezing all the earlier layers. Training is then continued with the top few layers unfrozen so that they can be fine-tuned to the new dataset.

The example below uses the Dogs versus Cats dataset from [Kaggle](https://www.kaggle.com/c/dogs-vs-cats), a website that hosts a large number of open datasets. We have downloaded the dataset for you. You can go straight to unzipping the training data.

The fine-tuning code was taken from the [Keras](https://keras.io/applications/) website.

- Use the **GPU T4** for this exercise
(Observe the GPU utilization while the model trains)

In [ ]:
!wget https://pdl-doulos.s3.us-west-2.amazonaws.com/train.zip

In [ ]:
import os
import zipfile

# Define names
target_dir = 'dogs_vs_cats'
zip_file_name = 'train.zip'

# 1. Create the directory 
if not os.path.exists(target_dir):
    os.makedirs(target_dir)
    print(f"created directory: {target_dir}")
else:
    print(f"Directory '{target_dir}' already exists.")

# 2. Extract the file 
if os.path.exists(zip_file_name):
    print(f"Extracting {zip_file_name}... please wait.")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print(f"Successfully unzipped to: {target_dir}")
else:
    print(f"Error: {zip_file_name} not found in the current directory.")

Define a function to construct and return one batch of training or test data. The dataset is first shuffled. The images in each batch are normalized before being returned.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
#from scipy import misc
from PIL import Image

BASE_DIR = os.getcwd()
path = os.path.join(BASE_DIR, 'dogs_vs_cats', 'train')

print('Reading data from', path)
file_list = os.listdir(path)
random.shuffle(file_list)
n = len(file_list)
n_test = n // 5
n_train = n - n_test
print('Dataset size =', n_train, 'training images and', n_test, 'test images')

def next_batch(imsize, test=False, batch_size=128):
    if test:
        n_images = n_test
    else:
        n_images = batch_size
    x = np.empty((n_images, imsize, imsize, 3)).astype('float32')
    y = np.empty((n_images, 2)).astype('float32')
    for i in range(n_images):
        if test:
            r = random.randrange(0,n_test)
        else:
            r = random.randrange(n_test,len(file_list))
        filename = file_list[r]
        file = path + '/' + filename
        im = imageio.imread(file)
        im = np.array(Image.fromarray(im).resize((imsize,imsize)))

        x[i] = im
        if filename[:3] == 'dog':
            y[i] = [1, 0]
        elif filename[:3] == 'cat':
            y[i] = [0, 1]
        else:
            print('Unexpected file name', filename)

        #plt.imshow(im)
        #plt.show()
        #print('Dog' if y[i][0] == 1 else 'Cat')

    # Normalize the data
    x = x / 255
    x = x - x.mean()
    return (x, y)

Load a pre-trained network, replace the final few layers of the network with untrained layers, freeze all the pre-trained weights, then train and evaluate the network. Then unfreeze a few of the pre-trained layers at the back end of the network and repeat.

In [ ]:
from tensorflow.keras.applications import ResNet50V2

# Load the ResNet50V2 model with pre-trained ImageNet weights
# Set include_top=False to exclude the final classification layers,
# as you are interested in the base model's layers.
model = ResNet50V2(weights='imagenet', include_top=False)

# Get the number of layers
num_layers = len(model.layers)

print(f"The ResNet50V2 base model has {num_layers} layers.")

# You can also iterate through the layers and print their names for more detail
#for i, layer in enumerate(model.layers):
       #print(f"Layer {i}: {layer.name}")

The following Transfer Learning cell takes time to execute. 
**Utilization of the T4 GPU goes very high when executing this cell.**

Observations to make:
- Click on the CPU usage bar during training
- Why do you think the CPU utilization is high thoughout the training, even though the GPU is doing the heavy lifting

### Solution

<details> 
    <summary>Here is our answer</summary>
     
    - The CPU utilization remains high as it is creating mini-batches for the GPU to use. 
    - This is done using the Python Generator function
    - The GPU utilization dips if it is waiting for the mini-batch to come from the CPU
                
</details>

In [ ]:
import tensorflow

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

tensorflow.keras.backend.clear_session()

# Load a large pre-trained model, omitting the final categorization layers
imsize = 256
unfreeze_after_layer = 152
base_model = ResNet50V2(weights='imagenet', include_top=False)

# Replace the final layers of the network with new global average pooling and dense layers
B = base_model.output
P = GlobalAveragePooling2D()(B)
S = Dense(2, activation='softmax')(P)

model = Model(inputs=base_model.input, outputs=S)

# Freeze all the convolutional feature layers
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

minibatch_size = 128

# Python generator to get the training dataset one minibatch at a time
def gen():
    while (True):
        yield next_batch(imsize, batch_size=minibatch_size)

# Get the test dataset
(x_test, y_test) = next_batch(imsize, test=True)

# Train just the new layers
model.fit(gen(), steps_per_epoch=n_train//minibatch_size, epochs=3)

loss_and_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy after training just the final layers: {100*loss_and_acc[1]:5.1f}%')

# Unfreeze a few convolutional layers
for layer in model.layers[:unfreeze_after_layer]:
   layer.trainable = False
for layer in model.layers[unfreeze_after_layer:]:
   layer.trainable = True

# Recompile the model for the unfreezing to take effect
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(gen(), steps_per_epoch=n_train//minibatch_size, epochs=3)

loss_and_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy after fine-tuning a few more layers: {100*loss_and_acc[1]:5.1f}%')
model.save('dogs_vs_cats_resnet.keras')